# 리포트 21 — 해석 PO 구 대비 구현오차는 kr 전 구간에서 λ/16 격자 0.201 dB · 생산 λ/12 격자 0.254 dB 안이다

> ### 한 일
> **구 후방산란의 닫힌형 기준해 둘과 이면각 닫힌형에 커널을 맞대 구현오차와 모형 간극을 따로 쟀다.**

### 결과
1. 해석 PO 구 대비 최대 편차가 kr 1 [^1]~100 [^2] 전 구간에서 λ/16 격자 0.201 [^3] · 생산 λ/12 격자 0.254 dB [^4] 다(입사 48 방향 [^5]).
2. 정확 Mie 대비 최대 편차는 λ/16 격자 6.73 [^6] · 생산 λ/12 격자 6.60 dB [^7] (kr=1) 이고, 이쪽이 PO 라는 모형 자체의 간극이다 — 격자를 209 배 [^8] 조여도 -0.054 dB [^9] 움직인다.
3. kr ≥ 30 산포는 λ/16 격자에서 해석 PO 대비 0.885% [^10] · Mie 대비 1.834% [^11] 이고, 생산 λ/12 격자에서 1.939% [^12] · 1.547% [^13] 다.
4. PEC 이면각 닫힌형 8πa²b²/λ² 와는 2회 반사에서 최대 0.556 dB [^14] 다 — 다중반사 위상이 맞는다는 뜻이다.
5. 기체 7 × 밴드 3 = 21 조합 [^15] 중 1 개 [^16] 가 Mie 기준 1 dB 문턱 아래에 놓이고, 그것이 실측 대상 DJI Mini 5 Pro [^17] 다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 기준해 둘 | 구에 대해서만 맥스웰 방정식이 그대로 풀리는 **정확 Mie** 와, 같은 구에 PO 근사를 적용해 손으로 푼 **해석 PO** 다 (`benchmark/mie_pec_sphere.py:98`, `:127`) |
| 왜 둘인가 | (커널 − Mie) = (커널 − 해석 PO) + (해석 PO − Mie) 다. 앞항은 **구현오차**(격자를 조이면 준다), 뒷항은 **모형오차**(격자로는 안 준다) |
| 다중반사 | 직각 이면각 이등분선 입사의 닫힌형 8πa²b²/λ² 와 변 길이 4점에서 맞댄다(`benchmark/verify_sbr_defect_fixes.py`, λ/12 격자) |
| 자기검사 | 상반성 σ(û_i,û_s)=σ(û_s,û_i) 위반을 기체에서 잰다 — 정리 위반이 곧 모형오차다 |

### 재현

```bash
PYTHONPATH=src python benchmark/sbr_kr_sweep.py
PYTHONPATH=src python benchmark/verify_sbr_defect_fixes.py
PYTHONPATH=src python src/build_part04_kernel.py
```

| | |
|---|---|
| 출력 | `outputs/sbr_kr_sweep.json`, `outputs/sbr_defect_fixes.json`, `outputs/report00_po_case.json`, `outputs/report02_derived.json` |
| 소요 | 약 1시간 (GPU 1장 — kr 스윕이 대부분이다) |
| 비고 | 두 기준해는 우리 출력이 아니라 과녁이다 |

---

## 과녁이 둘이고, 재는 것이 다르다

구 후방산란은 두 개의 **닫힌형 기준해**(근사 없이 식으로 바로 값이 나오는 답)를 갖는다 — 구에 대해서만 맥스웰 방정식이 그대로 풀리는 **정확 Mie** 와, 같은 구에 PO 근사를 적용해 손으로 푼 **해석 PO** 다. 둘 다 우리 출력이 아니라 과녁이다.

```
(커널 − Mie)  =  (커널 − 해석 PO)   +   (해석 PO − Mie)
                  ↑ 우리 수치오차          ↑ PO 모델 자체의 간극
```
커널이 PO 이므로 **수치 수렴의 과녁은 해석 PO** 이고, Mie 잔차는 PO 모델 자체의 간극이라는 두 번째 눈금이다. 둘을 나눠 두면 각각이 얼마인지 그대로 읽힌다.

## 일곱 기체가 놓인 자리에서 각각 얼마인가

![report02_f5_reference_gap](../outputs/figures/report02_f5_reference_gap.png)

**그림 1.** 일곱 기체가 놓인 kr 자리에서 우리 수치오차와 PO 모델의 간극은 각각 얼마인가?

## 두 눈금 — 격자 두 개에서

|  | 우리 수치오차 · 기준해 = 해석 PO | PO 모델의 간극 · 기준해 = 정확 Mie |
|---|---|---|
| 최대 편차 (kr=1..100) · λ/16 | 0.201 dB [^3] | 6.73 dB [^6] (kr=1) |
| 최대 편차 (kr=1..100) · 생산 λ/12 | 0.254 dB [^4] | 6.60 dB [^7] (kr=1) |
| kr≥30 산포 · λ/16 | 0.885% [^10] | 1.834% [^11] |
| kr≥30 산포 · 생산 λ/12 | 1.939% [^12] | 1.547% [^13] |
| 1 dB 안으로 드는 kr | 전 구간 (kr=1 [^1] 부터) | kr ≥ 9.06 [^18] |
| 0.5 / 0.2 dB 안으로 | 전 구간 | 15.16 [^19] / 30.87 [^20] |

격자를 두 줄로 적는 이유는 사슬이 둘이기 때문이다 — 커널 기본은 λ/12(`src/rcs_sbr.py` `DEFAULT_DIV`)이고, σ 앵커 사슬은 λ/16 으로 돈다(`benchmark/rcs_anchor.py` `raw_sigma_az(div=16)`).

## 검증 3층 — 무엇을 각각 재는가

| 층 | 과녁 | 무엇을 재나 | 결과 |
|---|---|---|---|
| ① | 해석 PO 구 | 커널 구현 | 최대 λ/16 0.201 [^21] · 생산 λ/12 0.254 dB [^4] |
| ② | PEC 구 Mie 정확해 | PO 라는 모형 | ka=1 에서 -6.58 dB [^22] · 광학영역 산포 λ/16 1.83 [^23] · 생산 λ/12 1.55% [^13] |
| ③ | 얇은 띠 2D EFIE MoM | 가는 특징 | 가장 가는 폭에서 TM -4.02 [^24] · TE +7.53 dB [^25] |
| + | PEC 이면각 닫힌형 | 다중반사 위상 | 2-bounce 최대 0.556 dB [^26] |
| + | 상반성 정리 | 정리 위반 = 모형오차 | 기체 최악 8.24 dB [^27] (같은 검사를 인쇄한 선행 0편) |

## 이면각 — 오목부에서 오는 항

**이면각**은 두 평판이 90° 로 맞붙은 표준 형상이고, **PEC** 는 전기를 완벽히 통하는 이상적 금속이다. 직각 이면각의 이등분선 입사는 σ = 8πa²b²/λ² 로 닫혀 있다. 2회 반사를 켜고 변 길이 4점에서 그 값과 맞댔다(3.5 GHz, λ/12 격자).

| 변 a [m] | 해석해 [dBsm] | 1회 반사 [dBsm] | 2회 반사 [dBsm] | 오차 [dB] |
|---|---|---|---|---|
| 0.15 | 2.39 | -14.63 | 2.95 | +0.556 |
| 0.20 | 7.39 | -13.79 | 7.85 | +0.466 |
| 0.30 | 14.43 | -118.33 | 14.51 | +0.076 |
| 0.40 | 19.43 | -7.76 | 19.32 | -0.108 |

출처 [^28]

1회/2회 열이 오목부에서 오는 항이 어디에 있는지 그대로 보여준다. 셋째 줄(a = 0.30 m)의 1회 반사는 -118.3 dBsm [^29] 로 무너진 널이고, 그 줄의 작은 오차는 그 우연한 상쇄 위에 앉아 있다 — 이 표에서 인용할 값은 4점의 **최대** 0.556 dB [^14] 다.

같은 스크립트가 매끄러운 기준체도 함께 잰다 — 구는 λ/16 격자에서 해석 PO 대비 -0.021 dB [^30], 평판은 λ/10 격자에서 -0.011 dB [^31] 다.

## 이 눈금이 드론에 그대로 걸리는가

기체 7 × 밴드 3 = 21 조합 [^15] 중 1 개 [^16] 가 Mie 기준 1 dB 문턱 아래에 놓이고, 그것이 실측 대상 DJI Mini 5 Pro [^17] 다.

⚠ **이 kr 눈금은 매끄러운 구에서만 맞는 눈금이다** — 구는 몸 전체가 하나의 넓은 곡면이지만 드론은 얇은 판과 가는 막대의 모음이라, PO 가 어긋나는 자리를 정하는 것은 기체 전체 크기가 아니라 **부품 하나의 폭이 파장에 비해 얼마나 넓은가** 다. 그 세 번째 눈금이 [편 22 «PO 유효 무릎을 부품 폭으로 옮기면 어느 부품이 어느 밴드에서 떨어지는지가 보인다»](22_po-knee.ipynb) 다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 평판·이면각 표준체로 같은 kr 스윕을 돌린다 | 얇고 모서리 많은 표적에서의 PO 간극 문턱이 선다 | `benchmark/verify_sbr_defect_fixes.py` 의 두 닫힌형 재사용 |
| 부품 폭 눈금으로 옮겨 우리 세 밴드가 어디에 서는지 읽는다 | 어느 부품이 어느 밴드에서 무릎 아래인지가 확정된다 | [편 22 «PO 유효 무릎을 부품 폭으로 옮기면 어느 부…»](22_po-knee.ipynb) |
| 상용 솔버 한 대와 같은 형상에서 교차검증한다 | 구·이면각 밖의 형상에서 구현오차가 확정된다 | `OPENSOURCE.md` — RadarSimPy 교차검증 항목 |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 31개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/sbr_kr_sweep.json` | `summary_div16.kr_min` | 1 |
| [^2] | `outputs/report00_po_case.json` | `s3_validation.layer1_analytic_po_convergence.kr_sweep_kr_max` | 100 |
| [^3] | `outputs/sbr_kr_sweep.json` | `summary_div16.max_abs_db_vs_po` | 0.2006 |
| [^4] | `outputs/sbr_kr_sweep.json` | `summary_div12.max_abs_db_vs_po` | 0.2544 |
| [^5] | `outputs/report00_po_case.json` | `s3_validation.layer1_analytic_po_convergence.kr_sweep_n_incidence` | 48 |
| [^6] | `outputs/sbr_kr_sweep.json` | `summary_div16.max_abs_db_vs_mie` | 6.729 |
| [^7] | `outputs/sbr_kr_sweep.json` | `summary_div12.max_abs_db_vs_mie` | 6.599 |
| [^8] | `outputs/report00_po_case.json` | `s3_validation.layer1_analytic_po_convergence.sphere_ka1_grid_refine_factor` | 209.4 |
| [^9] | `outputs/report00_po_case.json` | `s3_validation.layer2_pec_sphere_mie.improvement_from_refining_grid_db` | -0.05373 |
| [^10] | `outputs/sbr_kr_sweep.json` | `summary_div16.std_sbr_over_po_pct_kr_ge30` | 0.8855 |
| [^11] | `outputs/sbr_kr_sweep.json` | `summary_div16.std_sbr_over_mie_pct_kr_ge30` | 1.834 |
| [^12] | `outputs/sbr_kr_sweep.json` | `summary_div12.std_sbr_over_po_pct_kr_ge30` | 1.939 |
| [^13] | `outputs/sbr_kr_sweep.json` | `summary_div12.std_sbr_over_mie_pct_kr_ge30` | 1.547 |
| [^14] | `outputs/sbr_defect_fixes.json` | `d3_multibounce_phase.max_abs_err_db` | 0.5563 |
| [^15] | `outputs/report02_derived.json` | `electrical.n_airframe_band` | 21 |
| [^16] | `outputs/report02_derived.json` | `electrical.n_below_po_1db` | 1 |
| [^17] | `outputs/report02_derived.json` | `electrical.kr_min_name` | DJI Mini 5 Pro |
| [^18] | `outputs/report02_derived.json` | `po_floor.kr_below_1p0_db` | 9.06 |
| [^19] | `outputs/report02_derived.json` | `po_floor.kr_below_0p5_db` | 15.16 |
| [^20] | `outputs/report02_derived.json` | `po_floor.kr_below_0p2_db` | 30.87 |
| [^21] | `outputs/report00_po_case.json` | `s3_validation.layer1_analytic_po_convergence.kr_sweep_max_abs_db_vs_po_div16` | 0.2006 |
| [^22] | `outputs/report00_po_case.json` | `s3_validation.layer2_pec_sphere_mie.po_minus_mie_at_ka1_db` | -6.584 |
| [^23] | `outputs/report00_po_case.json` | `s3_validation.layer2_pec_sphere_mie.kr_sweep_std_pct_vs_mie_kr_ge30_div16` | 1.834 |
| [^24] | `outputs/report00_po_case.json` | `s3_validation.layer3_thin_plate_2d_mom.po_minus_tm_at_0p15lam_db` | -4.022 |
| [^25] | `outputs/report00_po_case.json` | `s3_validation.layer3_thin_plate_2d_mom.po_minus_te_at_0p15lam_db` | 7.535 |
| [^26] | `outputs/report00_po_case.json` | `s3_validation.layer4_dihedral_multibounce.max_abs_err_2bounce_db` | 0.5563 |
| [^27] | `outputs/report00_po_case.json` | `s3_validation.layer5_reciprocity_selfcheck.drone_worst_violation_db` | 8.237 |
| [^28] | `outputs/sbr_defect_fixes.json` | `d3_multibounce_phase.rows` | (4행 표) |
| [^29] | `outputs/report00_po_case.json` | `s3_validation.layer4_dihedral_multibounce.a03_sbr_1bounce_dbsm` | -118.3 |
| [^30] | `outputs/sbr_defect_fixes.json` | `d3_multibounce_phase.sphere_and_plate.sphere_vs_po_db.sphere_lam/16_vs_po` | -0.02084 |
| [^31] | `outputs/sbr_defect_fixes.json` | `d3_multibounce_phase.sphere_and_plate.plate_db.plate_lam/10` | -0.01114 |